# MUN-FRL: compact camera + IMU dataset for OpenVINS

This Colab notebook downloads an official MUN-FRL ROS 1 bag into the ephemeral `/content` disk, inspects its topics, and converts only the streams needed by the INS/OpenVINS propagation experiment into a compressed ROS 2 bag.

Estimator inputs: nadir monochrome camera and IMU. The MUN-FRL sample stores the mono stream as `sensor_msgs/CompressedImage`; it remains compressed here and will be decoded to `sensor_msgs/Image` by the downstream Jazzy replay adapter. RTK/PPK/FRL data are retained only as offline ground truth. LiDAR, color images, the front camera, GNSS corrections, and magnetometer data are excluded. `ins_only` will later mean IMU propagation during an artificial camera blackout after VIO initialization; this notebook does not construct a standalone INS model.

Sources: [MUN-FRL downloads](https://mun-frl-vil-dataset.readthedocs.io/en/latest/), [sensor topics](https://mun-frl-vil-dataset.readthedocs.io/en/latest/sensors.html), and [rosbags conversion API](https://ternaris.gitlab.io/rosbags/api/rosbags.convert.html).

In [2]:
# Pinned tools; no ROS installation is required for ROS 1 -> ROS 2 conversion.
%pip install -q gdown==5.2.0 rosbags==0.11.5


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.9/144.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 8.7 MB/s eta 0:00:00


## 1. Configuration

Start with the 6 GB official Bell 412 sample. It is small enough to validate the workflow before downloading the 27.2 GB `quarry1` dataset. To switch later, change `DATASET_ID` to `quarry1`. The source bag remains on Colab's temporary disk; only the filtered ROS 2 bag and manifest are copied to Google Drive.

In [4]:
from pathlib import Path
import hashlib
import json
import platform
import shutil
import sys
from datetime import datetime, timezone

DATASETS = {
    # Official CCECE 2025 sample linked by the MUN-FRL documentation.
    "bell412_dataset1_sample": {
        "download_kind": "file",
        "drive_id": "1dxZcWZ_J7QEXkDjaHdePMfO5him3EZuf",
        "expected_source_gb": 6.0,
    },
    # Official quarry1 folder. Select this only after the sample succeeds.
    "quarry1": {
        "download_kind": "folder",
        "drive_id": "1-cN8PKBhh5REuKsP69V0s5cf-PcBO_eR",
        "expected_source_gb": 27.2,
    },
}

DATASET_ID = "bell412_dataset1_sample"
DOWNLOAD_FROM_OFFICIAL_SOURCE = True
HASH_SOURCE_BAGS = True

WORK_ROOT = Path("/content/munfrl_work")
RAW_ROOT = WORK_ROOT / "raw" / DATASET_ID
OUTPUT_BAG = WORK_ROOT / "filtered" / f"{DATASET_ID}_openvins"
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/dis-proj/munfrl_filtered")

dataset = DATASETS[DATASET_ID]
print(json.dumps({"dataset_id": DATASET_ID, **dataset}, indent=2))


{
  "dataset_id": "bell412_dataset1_sample",
  "download_kind": "file",
  "drive_id": "1dxZcWZ_J7QEXkDjaHdePMfO5him3EZuf",
  "expected_source_gb": 6.0
}


## 2. Mount Drive and verify ephemeral disk space

The check reserves room for both the original ROS 1 bag and a filtered ROS 2 copy. Colab disk capacity is not guaranteed, so the notebook stops before downloading if the current runtime is too small.

In [6]:
from google.colab import drive
drive.mount("/content/drive")

usage = shutil.disk_usage("/content")
free_gb = usage.free / 10**9
# Filtered data can approach the source size if the sample already excludes LiDAR.
required_gb = dataset["expected_source_gb"] * 2 + 5.0
print(f"Free ephemeral disk: {free_gb:.1f} GB")
print(f"Conservative requirement: {required_gb:.1f} GB")
if free_gb < required_gb:
    raise RuntimeError(
        "This Colab runtime does not have enough ephemeral disk. "
        "Reconnect to a runtime with more disk or use the smaller sample.")

RAW_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_BAG.parent.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive
Free ephemeral disk: 94.1 GB
Conservative requirement: 17.0 GB


## 3. Download the official source into `/content`

If Google Drive applies a download quota, download the file through the browser into your own Drive, set `DOWNLOAD_FROM_OFFICIAL_SOURCE = False`, and place the `.bag` file under `RAW_ROOT` before continuing. Do not copy the full source into the final output directory.

In [4]:
import gdown

if DOWNLOAD_FROM_OFFICIAL_SOURCE:
    if any(RAW_ROOT.rglob("*.bag")):
        print("A source .bag already exists; skipping download.")
    elif dataset["download_kind"] == "file":
        target = RAW_ROOT / f"{DATASET_ID}.bag"
        result = gdown.download(id=dataset["drive_id"], output=str(target), quiet=False)
        if result is None:
            raise RuntimeError("Google Drive download failed or hit a quota.")
    else:
        result = gdown.download_folder(
            id=dataset["drive_id"], output=str(RAW_ROOT),
            quiet=False, remaining_ok=True,
        )
        if not result:
            raise RuntimeError("Google Drive folder download failed or hit a quota.")

SOURCE_BAGS = sorted(RAW_ROOT.rglob("*.bag"))
if not SOURCE_BAGS:
    raise FileNotFoundError(f"No ROS 1 .bag files found under {RAW_ROOT}")

for path in SOURCE_BAGS:
    print(f"{path}: {path.stat().st_size / 10**9:.2f} GB")


Downloading...
From (original): https://drive.google.com/uc?id=1dxZcWZ_J7QEXkDjaHdePMfO5him3EZuf
From (redirected): https://drive.google.com/uc?id=1dxZcWZ_J7QEXkDjaHdePMfO5him3EZuf&confirm=t&uuid=0cc2755c-27a1-48f0-844c-14e39f35813a
To: /content/munfrl_work/raw/bell412_dataset1_sample/bell412_dataset1_sample.bag
100%|██████████| 6.44G/6.44G [01:23<00:00, 76.9MB/s]

/content/munfrl_work/raw/bell412_dataset1_sample/bell412_dataset1_sample.bag: 6.44 GB


## 4. Inspect topics before filtering

This is a required scientific check. Topic names can differ between MUN-FRL releases, and the sample may not contain every stream present in a full bag.

In [5]:
from rosbags.highlevel import AnyReader

with AnyReader(SOURCE_BAGS) as reader:
    print(f"Duration: {reader.duration / 1e9:.2f} s")
    print(f"Messages: {reader.message_count:,}")
    print("\nTopic | Type | Messages")
    print("--- | --- | ---:")
    for topic, info in sorted(reader.topics.items()):
        print(f"{topic} | {info.msgtype} | {info.msgcount:,}")


Duration: 315.58 s
Messages: 581,458

Topic | Type | Messages
--- | --- | ---:
/Odometry | nav_msgs/msg/Odometry | 2,969
/camera/camera_info | sensor_msgs/msg/CameraInfo | 4,739
/camera/image_color/compressed | sensor_msgs/msg/CompressedImage | 4,739
/camera/image_mono_compressed/compressed | sensor_msgs/msg/CompressedImage | 4,586
/camera/image_mono_compressed/compressed/parameter_descriptions | dynamic_reconfigure/msg/ConfigDescription | 1
/camera/image_mono_compressed/compressed/parameter_updates | dynamic_reconfigure/msg/Config | 1
/cloud_registered | sensor_msgs/msg/PointCloud2 | 2,968
/fix | sensor_msgs/msg/NavSatFix | 1,578
/globalEstimator/car_model | visualization_msgs/msg/MarkerArray | 2,272
/globalEstimator/frl_path | nav_msgs/msg/Path | 2,272
/globalEstimator/global_path | nav_msgs/msg/Path | 2,272
/globalEstimator/gps_path | nav_msgs/msg/Path | 2,272
/globalEstimator/ppk_path | nav_msgs/msg/Path | 2,272
/imu/data | sensor_msgs/msg/Imu | 125,933
/imu/data_stamped | xsens_mt

## 5. Select the minimal OpenVINS and evaluation streams

The selector keeps one nadir monochrome image stream and one standard IMU stream. `/fix_frl` is preferred because it is documented as 6DoF ground truth; `/fix_ppk` and `/fix` are position-only fallbacks and require a reduced validation model. The benchmark sample also contains global-estimator PPK/FRL path topics; they are retained only as offline references and must not be fed into OpenVINS. Existing `/Odometry`, `/path`, and `/vins_estimator/*` outputs are not treated as ground truth. RGB is intentionally not copied into this VIO bag; it can later be exported separately for visual map localization.

In [6]:
with AnyReader(SOURCE_BAGS) as reader:
    available = set(reader.topics)
    topic_types = {topic: info.msgtype for topic, info in reader.topics.items()}

def first_available(label, candidates, *, required=True):
    selected = next((topic for topic in candidates if topic in available), None)
    if selected is None and required:
        raise RuntimeError(
            f"Missing required {label}. Tried: {', '.join(candidates)}")
    return selected

camera_topic = first_available(
    "nadir mono camera", [
        "/camera/image_mono",
        "/camera/image_raw",
        "/camera/image_mono_compressed/compressed",
    ])
# OpenVINS consumes sensor_msgs/Imu. The sample's /imu/data_stamped uses
# the custom xsens_mti_driver/IMUwithTimeRef type and is not replayed.
imu_topic = first_available("standard IMU", ["/imu/data"])
ground_truth_topic = first_available(
    "ground truth", ["/fix_frl", "/fix_ppk", "/fix"])

optional_candidates = [
    "/imu/time_ref",
    "/imu/time_ref_cam",
    "/imu/time_ref_pps",
    "/camera/camera_info",
    "/tf_static",
]
offline_reference_candidates = [
    "/globalEstimator/ppk_path",
    "/globalEstimator/frl_path",
]
offline_reference_topics = tuple(
    topic for topic in offline_reference_candidates if topic in available)
SELECTED_TOPICS = [camera_topic, imu_topic, ground_truth_topic]
SELECTED_TOPICS.extend(topic for topic in optional_candidates if topic in available)
SELECTED_TOPICS.extend(offline_reference_topics)
SELECTED_TOPICS = tuple(dict.fromkeys(SELECTED_TOPICS))
camera_msgtype = topic_types[camera_topic]
imu_msgtype = topic_types[imu_topic]
if imu_msgtype != "sensor_msgs/msg/Imu":
    raise RuntimeError(f"OpenVINS requires sensor_msgs/msg/Imu, found {imu_msgtype}")
if camera_msgtype not in {"sensor_msgs/msg/Image", "sensor_msgs/msg/CompressedImage"}:
    raise RuntimeError(f"Unsupported camera type: {camera_msgtype}")

print("Selected topics:")
for topic in SELECTED_TOPICS:
    print(f"  {topic}")

if ground_truth_topic != "/fix_frl":
    print("WARNING: selected ground truth is position-only. "
          "Do not fit yaw/attitude errors from this run.")

excluded_heavy = sorted(topic for topic in available if any(token in topic.lower() for token in (
    "velodyne", "pointcloud", "image_color", "front_camera")))
print("\nHeavy streams excluded:")
for topic in excluded_heavy:
    print(f"  {topic}")


Selected topics:
  /camera/image_mono_compressed/compressed
  /imu/data
  /fix
  /imu/time_ref
  /imu/time_ref_cam
  /imu/time_ref_pps
  /camera/camera_info
  /tf_static
  /globalEstimator/ppk_path
  /globalEstimator/frl_path

Heavy streams excluded:
  /camera/image_color/compressed
  /velodyne_cloud_registered
  /velodyne_points


## 6. Convert the selected ROS 1 topics to compressed ROS 2 Jazzy format

The output uses rosbag2 SQLite storage, format version 9, and Zstandard message compression. Message compression is required so Jazzy can replay the bag directly from a read-only NTFS mount without creating a large decompressed database beside it. The conversion refuses to overwrite an existing output directory.

In [7]:
from rosbags.convert import convert
from rosbags.typesys import Stores, get_typestore

if OUTPUT_BAG.exists():
    raise FileExistsError(f"Refusing to overwrite {OUTPUT_BAG}")

convert(
    srcs=SOURCE_BAGS,
    dst=OUTPUT_BAG,
    dst_storage="sqlite3",
    dst_version=9,
    compress="zstd",
    compress_mode="message",
    default_typestore=get_typestore(Stores.ROS1_NOETIC),
    typestore=get_typestore(Stores.ROS2_JAZZY),
    exclude_topics=(),
    include_topics=SELECTED_TOPICS,
    exclude_msgtypes=(),
    include_msgtypes=(),
)
print(f"Created {OUTPUT_BAG}")


Created /content/munfrl_work/filtered/bell412_dataset1_sample_openvins


## 7. Validate the filtered bag and write provenance

Validation checks topic retention, message counts, duration, output size, and SHA-256 fingerprints before anything is copied to persistent storage.

In [8]:
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

with AnyReader([OUTPUT_BAG]) as reader:
    output_topics = set(reader.topics)
    missing = set(SELECTED_TOPICS) - output_topics
    if missing:
        raise RuntimeError(f"Filtered bag is missing topics: {sorted(missing)}")
    output_summary = {
        topic: {"msgtype": info.msgtype, "message_count": info.msgcount}
        for topic, info in sorted(reader.topics.items())
    }
    duration_s = reader.duration / 1e9
    message_count = reader.message_count

output_files = sorted(path for path in OUTPUT_BAG.rglob("*") if path.is_file())
output_size = sum(path.stat().st_size for path in output_files)
manifest = {
    "schema_version": 1,
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_id": DATASET_ID,
    "source": {
        "provider": "MUN-FRL",
        "official_drive_id": dataset["drive_id"],
        "bags": [
            {
                "name": path.name,
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path) if HASH_SOURCE_BAGS else None,
            }
            for path in SOURCE_BAGS
        ],
    },
    "selection": {
        "camera_topic": camera_topic,
        "camera_msgtype": camera_msgtype,
        "camera_requires_runtime_decode": camera_msgtype == "sensor_msgs/msg/CompressedImage",
        "imu_topic": imu_topic,
        "imu_msgtype": imu_msgtype,
        "ground_truth_topic": ground_truth_topic,
        "ground_truth_position_only": ground_truth_topic != "/fix_frl",
        "offline_reference_topics": list(offline_reference_topics),
        "topics": list(SELECTED_TOPICS),
        "lidar_used": False,
        "standalone_ins_model": False,
    },
    "output": {
        "format": "rosbag2 sqlite3 v9 zstd-message",
        "duration_s": duration_s,
        "message_count": message_count,
        "size_bytes": output_size,
        "topics": output_summary,
        "files": [
            {
                "relative_path": str(path.relative_to(OUTPUT_BAG)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
            for path in output_files
        ],
    },
    "environment": {
        "python": sys.version,
        "platform": platform.platform(),
        "rosbags_version": __import__("importlib.metadata").metadata.version("rosbags"),
    },
}

manifest_path = OUTPUT_BAG.parent / f"{DATASET_ID}_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print(json.dumps({
    "duration_s": duration_s,
    "message_count": message_count,
    "output_size_gb": output_size / 10**9,
    "topics": list(output_summary),
}, indent=2))


{
  "duration_s": 315.583256949,
  "message_count": 276886,
  "output_size_gb": 0.823478123,
  "topics": [
    "/camera/camera_info",
    "/camera/image_mono_compressed/compressed",
    "/fix",
    "/globalEstimator/frl_path",
    "/globalEstimator/ppk_path",
    "/imu/data",
    "/imu/time_ref",
    "/imu/time_ref_cam",
    "/imu/time_ref_pps",
    "/tf_static"
  ]
}


## 8. Copy only the compact artifact to Google Drive

The raw MUN-FRL bag is deliberately not persisted. If this cell is interrupted, remove the incomplete destination in Google Drive before retrying.

In [9]:
persistent_bag = DRIVE_OUTPUT_ROOT / OUTPUT_BAG.name
persistent_manifest = DRIVE_OUTPUT_ROOT / manifest_path.name
if persistent_bag.exists() or persistent_manifest.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing Drive output for {DATASET_ID}")

shutil.copytree(OUTPUT_BAG, persistent_bag)
shutil.copy2(manifest_path, persistent_manifest)
print(f"Persistent ROS 2 bag: {persistent_bag}")
print(f"Manifest: {persistent_manifest}")
print("Next: copy this compact bag to the Windows NTFS partition and validate it with ros2 bag info under Jazzy.")


Persistent ROS 2 bag: /content/drive/MyDrive/dis-proj/munfrl_filtered/bell412_dataset1_sample_openvins
Manifest: /content/drive/MyDrive/dis-proj/munfrl_filtered/bell412_dataset1_sample_manifest.json
Next: copy this compact bag to the Windows NTFS partition and validate it with ros2 bag info under Jazzy.


## 9. One-time migration of an older `zstd-file` artifact

Run this section only if the compact bag was generated by an older version of this notebook and its `metadata.yaml` says `compression_mode: file`. It repacks the existing Drive artifact without downloading the 6 GB source again. The resulting `zstd-message` bag can be replayed by Jazzy directly from a read-only NTFS mount.

In [7]:
from rosbags.highlevel import AnyReader
from rosbags.rosbag2 import Writer, CompressionFormat, CompressionMode

file_compressed_bag = DRIVE_OUTPUT_ROOT / f"{DATASET_ID}_openvins"
message_bag = Path("/content/munfrl_repack") / f"{DATASET_ID}_openvins_message"
persistent_message_bag = DRIVE_OUTPUT_ROOT / message_bag.name
if not file_compressed_bag.is_dir():
    raise FileNotFoundError(file_compressed_bag)
if message_bag.exists() or persistent_message_bag.exists():
    raise FileExistsError("Refusing to overwrite an existing message-compressed bag")
message_bag.parent.mkdir(parents=True, exist_ok=True)

reader = AnyReader([file_compressed_bag])
writer = Writer(message_bag, version=9)
writer.set_compression(CompressionMode.MESSAGE, CompressionFormat.ZSTD)
with reader, writer:
    expected_message_count = reader.message_count
    output_connections = {
        connection.id: writer.add_connection(
            connection.topic,
            connection.msgtype,
            msgdef=connection.msgdef.data,
            rihs01=connection.digest,
            serialization_format=connection.ext.serialization_format,
            offered_qos_profiles=connection.ext.offered_qos_profiles,
        )
        for connection in reader.connections
    }
    for connection, timestamp, rawdata in reader.messages():
        writer.write(output_connections[connection.id], timestamp, rawdata)

with AnyReader([message_bag]) as check:
    if check.message_count != expected_message_count:
        raise RuntimeError(f"Unexpected repacked message count: {check.message_count}")
shutil.copytree(message_bag, persistent_message_bag)
print(f"Message-compressed ROS 2 bag: {persistent_message_bag}")
print("Download this *_openvins_message directory for the Ubuntu replay.")


Message-compressed ROS 2 bag: /content/drive/MyDrive/dis-proj/munfrl_filtered/bell412_dataset1_sample_openvins_message
Download this *_openvins_message directory for the Ubuntu replay.
